# 02. 같은 표본의 유전자·순위·경로 표현

같은 RNA를 세 가지로 표현합니다. (A1) 유전자별 log2 값, (A1r) 표본 안에서의 유전자 순위, (A3) Hallmark 50개 집합의 점수. 경로 점수는 유전자 집합의 발현 요약이며, 단백질 활성이나 인과적 경로 활성을 직접 잰 값이 아닙니다.

In [ ]:
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "configs", "pathway_sensitivity_a1")):
    parent = os.path.dirname(root)
    assert parent != root, "저장소 안에서 실행해 주세요."
    root = parent
sys.path.insert(0, os.path.join(root, "src", "pathway_sensitivity_a1"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import core
cfg = core.load_config()
print("저장소:", root)
print("입력 캐시:", core.cache_dir(cfg))

In [ ]:
limore = core.load_study(cfg, "LIMORE")
axis = pd.read_csv(os.path.join(core.MANIFESTS, "gene_axis.tsv"), sep="\t")
i = limore.models.index("LIMORE__Huh7")
print("A1  유전자 표현:", limore.expr.shape[1], "차원")
print("A1r 순위 표현:", limore.rank.shape[1], "차원")
print("A3  경로 표현:", limore.hallmark.shape[1], "차원")

## 순위 표현

표본 하나 안에서 유전자를 발현 순으로 줄 세우고 전체 유전자 수로 나눕니다. 다른 표본의 값은 전혀 쓰지 않습니다.

In [ ]:
genes = ["ALB", "AFP", "GPC3", "TP53", "MKI67"]
idx = [list(axis.symbol).index(g) for g in genes]
print(pd.DataFrame({"log2 FPKM": limore.expr[i, idx], "상대 순위": limore.rank[i, idx]}, index=genes))
assert np.allclose(core.relative_rank(limore.expr[[i]]), limore.rank[[i]])   # 캐시와 같은 계산

## 경로 점수 (singscore)

집합에 속한 유전자들의 평균 순위를 이론적 최소·최대로 정규화한 값입니다. 저장된 점수는 Bioconductor singscore로 계산했습니다. 아래에서 한 집합을 numpy로 직접 계산해 같은지 확인합니다. 이 셀은 서버의 Hallmark GMT 파일을 읽습니다.

In [ ]:
gmt = core.read_gmt(core.rpath(cfg["paths"]["hallmark_gmt"]))
name = "HALLMARK_G2M_CHECKPOINT"
pos = {s: k for k, s in enumerate(axis.symbol)}
members = np.array([pos[g] for g in dict.fromkeys(gmt[name]) if g in pos])
ranks = core.min_rank(limore.expr[[i]])
mine = core.singscore_up(ranks, members)[0]
stored = limore.hallmark[i, limore.sets.index(name)]
print(name, "유전자", len(members), "개  numpy:", round(mine, 6), " 저장값:", round(stored, 6))
assert abs(mine - stored) < 1e-9

In [ ]:
scores = pd.Series(limore.hallmark[i], index=limore.sets).sort_values()
print("이 표본에서 점수가 가장 낮은 집합:\n", scores.head(5))
print("가장 높은 집합:\n", scores.tail(5))

## 음성 대조: 유전자 이름을 섞은 경로

집합의 크기와 집합끼리의 겹침은 그대로 두고 유전자 이름만 바꾼 20개 대조가 있습니다. 실제 Hallmark가 이 대조보다 예측에 도움이 되는지를 보는 용도입니다.

In [ ]:
perm = core.load_perm_scores(cfg, "LIMORE", cfg["a4"]["seeds"][0], limore.models, limore.sets)
print("실제 집합 점수의 모델 간 표준편차 평균:", round(limore.hallmark.std(axis=0).mean(), 4))
print("섞은 집합 점수의 모델 간 표준편차 평균:", round(perm.std(axis=0).mean(), 4))